# 필기체 숫자 인식

지난 학기에 Categorical Cross Entropy의 loss 펑션과 SGD 옵티마저를 이용하여 Numpy 기반 신경망을 구현 하였다.

이번에는 Optimizer의 이해를 높이기 위해, 몇가지 추가 Optimizer를 구현해 보자.

## MNIST Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from keras.utils import to_categorical

from keras.datasets import mnist     # MNIST dataset is included in Keras

MNIST 데이터를 읽어서 트레인 데이터와 테스트 데이터로 각각 로딩한다.

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()


In [ ]:
# Let's backup the test image data for later use
import copy
X_train_copy = copy.deepcopy(X_train) # backup the train data
X_test_copy = copy.deepcopy(X_test) # backup the test data

In [ ]:
X_train = X_train.reshape(60000, 784) # reshape 60,000 28 x 28 matrices into 60,000 784-length vectors.
X_test = X_test.reshape(10000, 784)   # reshape 10,000 28 x 28 matrices into 10,000 784-length vectors.

X_train = X_train.astype('float32')   # change integers to 32-bit floating point numbers
X_test = X_test.astype('float32')

X_train /= 255                        # normalize each value for each pixel for the entire vector for each input
X_test /= 255

nb_classes = 10
Y_train = to_categorical(y_train, nb_classes)
Y_test = to_categorical(y_test, nb_classes)

print("Training matrix shape", X_train.shape)
print("Testing matrix shape", X_test.shape)

In [ ]:

plt.rcParams['figure.figsize'] = (9,9) # Make the figures a bit bigger

for i in range(9):
    plt.subplot(3,3,i+1)
    num = random.randint(0, len(X_train))
    plt.imshow(X_train_copy[num], cmap='gray', interpolation='none')
    plt.title("Class {}".format(y_train[num]))

plt.tight_layout()

# NN build

![NeuralNetwork2](https://drive.google.com/uc?id=1_OIMh83I6OKTd7MQ_zljM9zOJ3mUWrAF)




This notebook implements a neural network for MNIST digit recognition using a **non-Object-Oriented, functional style**. All logic is contained within standalone functions.

The network's state (weights and biases) is managed in a dictionary called `parameters`, which is explicitly passed between functions.

### Network Architecture
- **Input Layer**: 784 neurons
- **Hidden Layer 1**: 16 neurons (ReLU)
- **Hidden Layer 2**: 16 neurons (ReLU)
- **Output Layer**: 10 neurons (Softmax)

## feed forward

In [ ]:
w_1    = np.random.rand(16, 784)   # (# of neurons, # of inputs)
b_1      = np.random.rand(16)
w_2    = np.random.rand(16, 16)
b_2      = np.random.rand(16)
w_o    = np.random.rand(10, 16)
b_o      = np.random.rand(10)

**Lets' check the shape of vectors**

In [ ]:
print(f"X_train shape  : {X_train.shape}")
print(f"weight 1 shape : {w_1.shape}")

weight_1과 X_train 을 dot product를 하기 위해서는 X_train을 transpose 해야 한다.

In [ ]:
z_1 = np.dot(w_1, X_train.T)
z_1 = z_1.T + b_1

In [ ]:
# Sigmoid: 어떤 숫자든 0과 1 사이의 값으로 압축하는 함수 (큰 숫자→1에 가깝게, 작은 숫자→0에 가깝게)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

activation of first hidden layer a1

In [ ]:
a_1 = sigmoid(z_1)

**Lets' check the shape of vectors**

In [ ]:
print(f"activation 1 shape  : {a_1.shape}")
print(f"weight 2 shape      : {w_2.shape}")

In [ ]:
z_2 = np.dot(w_2, a_1.T)
z_2 = z_2.T + b_2
a_2 = sigmoid(z_2)

In [ ]:
z_o = np.dot(w_o, a_2.T)
z_o = z_o.T + b_o

In [ ]:
# Softmax는 숫자 벡터를 확률 분포로 변환 (모든 값의 합 = 1).
def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return exp_x / np.sum(exp_x)

In [ ]:
output = softmax(z_o)

### 손실 구하기

In [ ]:
def cross_entropy_loss(y_actual, y_pred):
    # Avoid division by zero and log(0)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_actual * np.log(y_pred)) / len(y_actual)

Cross Entropy는 **두 확률 분포가 얼마나 다른가**를 측정하는 지표

Cross Entropy 공식
```python
-sum(실제값 × log(예측값))
```

보통 backpropgation 설명시에 다음과 같이 에러의 제곱합을 이용하는데,
```python
def mse_loss(y_actual, y_pred):
    return np.sum(np.square(y_actual - y_pred)) / len(y_actual)
```

 이는 제곱 즉 2차원 함수의 미분인 기울기를 시각화 하기 용이 하기 때문이다.


그런데 여기에서 Cross Entropy를 사용하는 이유는 Softmax와 Cross Entropy를 사용할때 미분이 간단해지기 때문이다.

In [ ]:
loss = cross_entropy_loss(Y_train, output)
print("Loss:", loss)

### 경사 구하기 & Backpropagation

#### 출력레이어 액티베이션에 대한 경사

In [ ]:
# Backpropagation
# Gradient of the output layer
d_output = output - Y_train # Derivative of cross-entropy loss with respect to output


print(d_output.shape)
d_output[0]

출력단에서 softmax 와 cross entropy 를 사용하면 미분이 $- (actual - prediction)$ 이 되어 $$d\_output = - (Y\_train - output) = output - Y\_train$$이 된다.

즉 이것은 $$\frac{\partial{\mathbf{L}}}{\partial{\mathbf{z_o}}}$$ 의 의미이며, z 값(softmax에 입력되는 값)에 따른 Loss의 변화를 말하는 것이다.

현재 손실의 의미:
* 0 : 완벽한 예측
* 음수: 이 클래스의 확률을 높여야 함 (정답인데 낮게 예측)
* 양수: 이 클래스의 확률을 낮춰야 함 (오답인데 높게 예측)

#### 출력레이어 가중치 곱 합계에 대한 경사

이제 출력단에서 가중치들이 바뀌면 Loss가 어떻게 바뀌는지를 나타내는 식
$\frac{\partial{L}}{\partial{W_o}}$ 를 살펴 보면 chain rule에 따라 다음과 같다.


$$
\frac{\partial{L}}{\partial{W_o}} = \frac{\partial{L}}{\partial{z}}\frac{\partial{z}}{\partial{W_o}}
$$
이것은 다음의 의미이다.


$$\text{(가중치에 대한 기울기)} = \text{평균} \left[ \text{(출력 에러)} \otimes \text{(입력값)} \right]$$
즉 $\frac{\partial{L}}{\partial{z}}$ 은 출력에러이고, $\frac{\partial{z}}{\partial{W_o}}$ 는 softmax에 대한 입력값이 된다.



여기에서 $\mathbf{a_{h2}}$는 hidden_2의 activation 을 나타 낼때 $z_{out}$ 즉 output layer의 가중치와 앞 레이어(hidden_2)와의 곱의 합은 다음과 같다.

$$
\mathbf{z_o} = \mathbf{W_o} \mathbf{a_{h2}} + \mathbf{b_o}
$$
이고 여기에서 수학적 미분은 즉 가중치($\mathbf{W_o}$)를 변경할때 $\mathbf{z}$의 변화의 기울기는 다음식과 같이 앞단의 activation 즉 $\mathbf{a_{h2}}$ 에 비례하게 되며 다음과 같이 나타낼 수 있다.

$$\frac{\partial \mathbf{z_o}}{\partial \mathbf{W}_o} = \mathbf{a_{h2}}$$
이를 원 식에 대입하면 다음과 같다.
$$\frac{\partial L}{\partial \mathbf{W}_o} = \frac{\partial L}{\partial \mathbf{z_o}} \cdot (\mathbf{a_{h2}})^T$$

여기에서 출력에러는 출력단의 뉴론 갯수(10)의 벡터이고 입력은 앞단(hidden2)의 뉴론 갯수(16) 이다. 좌변의 $\mathbf{W_o}$는 (10, 16)의 행렬이고, 이를 구하기 위해 출력에러와 입력의 외적을 구한 것이다.

따라서 다음과 같이 구한다.

bias는 입력에 곱해지지 않으므로 출력에러와 같으며, 이에 대한 평균으로 한다.


In [ ]:
# Gradient of the output layer weights and biases
d_weight_o = np.dot(d_output.T, a_2) / len(Y_train)
d_bias_o = np.sum(d_output, axis=0) / len(Y_train)

#### 히든레이어2의 액티베이션의 경사

다음으로 출력단 바로 앞의 레이어 즉 hidden_2의 activation에 대한 미분과 가중치 계산에 대한 미분을 살펴 보자

우선 sigmoid 식을 살펴보면
$$\sigma(\mathbf{z_{h2}})=  \frac{1}{1 + e^{-\mathbf{z_{h2}}}}$$
이고 이에 대한 미분은
$$\frac{\partial{\mathbf{a_{h2}}}}{\partial{\mathbf{z_{h2}}}} = \sigma(\mathbf{z_{h2}}) \cdot (1 - \sigma(\mathbf{z_{h2}}))$$ 이 된다. 이에 대한 코드는
```python
    (layer2 * (1 - layer2))
```
이 된다. 이는 hidden_2의 activation 함수가 주는 영향이다.


이제 hidden_2의 activation 이 Loss에 주는 영향을 다시 써보면
$$\frac{\partial{\mathbf{L}}}{\partial{\mathbf{z_{h2}}}} = \frac{\partial{\mathbf{L}}}{\partial{\mathbf{z_{o}}}} \cdot \frac{\partial{\mathbf{z_{o}}}}{\partial{\mathbf{z_{h2}}}} = \frac{\partial{\mathbf{L}}}{\partial{\mathbf{z_o}}} \cdot \frac{\partial{\mathbf{z_o}}}{\partial{\mathbf{a_{h2}}}} \cdot \frac{\partial{\mathbf{a_{h2}}}}{\partial{\mathbf{z_{h2}}}} = \frac{\partial{\mathbf{{L}}}}{\partial{\mathbf{z_o}}} \cdot \mathbf{W_o} \cdot \sigma(\mathbf{z_{h2}})(1 - \sigma(\mathbf{z_{h2}}))$$

hidden_2의 activation 이 Loss에 주는 영향을 나타내는 식은 다음 좌변과 같고 이는 체인룰에 따라 우변처럼 쓸수 있다.


In [ ]:
# Gradient of the second hidden layer
d_layer2 = np.dot(d_output, w_o) * (a_2 * (1 - a_2)) # Derivative through sigmoid activation

#### 히든레이어2의 가중치 곱 합계에 대한 경사

In [ ]:
# Gradient of the second hidden layer weights and biases
d_weight_2 = np.dot(d_layer2.T, a_1) / len(Y_train)
d_bias_2 = np.sum(d_layer2, axis=0) / len(Y_train)


#### 히든레이어1의 액티베이션의 경사

In [ ]:
# Gradient of the first hidden layer
d_layer1 = np.dot(d_layer2, w_2) * (a_1 * (1 - a_1)) # Derivative through sigmoid activation

#### 히든레이어1의 가중치 곱 합계에 대한 경사

In [ ]:
# Gradient of the first hidden layer weights and biases
d_weight_1 = np.dot(d_layer1.T, X_train) / len(Y_train)
d_bias_1 = np.sum(d_layer1, axis=0) / len(Y_train)


#### Gradients 정리

In [ ]:

print("Gradient of weight_1 shape:", d_weight_1.shape)
print("Gradient of bias_1 shape:", d_bias_1.shape)
print("Gradient of weight_2 shape:", d_weight_2.shape)
print("Gradient of bias_2 shape:", d_bias_2.shape)
print("Gradient of weight_o shape:", d_weight_o.shape)
print("Gradient of bias_o shape:", d_bias_o.shape)

In [ ]:
# Optimization step
learning_rate = 0.01

# Update weights and biases
w_1 = w_1 - learning_rate * d_weight_1
b_1 = b_1 - learning_rate * d_bias_1

w_2 = w_2 - learning_rate * d_weight_2
b_2 = b_2 - learning_rate * d_bias_2

w_o = w_o - learning_rate * d_weight_o
b_o = b_o - learning_rate * d_bias_o

feed forward 과정, Gradients 구하는 과정, 그리고 구해진 Gradient를 적용하여 가중치를 조정하는 Opitimization 과정을 살펴 보았다. 이게 원하는 가중치를 찾기 위한 과정이다. 이과정을 반복적으로 수행하기 위한 함수를 만들어 보자.

## 각 단계별 펑션 정의

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def softmax(x):
    # Handle both single samples and batches
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def cross_entropy_loss(y_actual, y_pred):
    # Avoid division by zero and log(0)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_actual * np.log(y_pred)) / len(y_actual)

In [ ]:
def build_network():
    model = dict()
    # Xavier Initialization for weights
    model['w_1'] = np.random.randn(16, 784) * np.sqrt(2 / (784 + 16))
    model['b_1'] = np.zeros(16)
    model['w_2'] = np.random.randn(16, 16) * np.sqrt(2 / (16 + 16))
    model['b_2'] = np.zeros(16)
    model['w_o'] = np.random.randn(10, 16) * np.sqrt(2 / (16 + 10))
    model['b_o'] = np.zeros(10)
    return model

In [ ]:
def feedforward(model, X_train):
    z_1 = np.dot(model['w_1'], X_train.T)
    z_1 = z_1.T + model['b_1']
    a_1 = sigmoid(z_1)
    z_2 = np.dot(model['w_2'], a_1.T)
    z_2 = z_2.T + model['b_2']
    a_2 = sigmoid(z_2)
    z_o = np.dot(model['w_o'], a_2.T)
    z_o = z_o.T + model['b_o']
    output = sigmoid(z_o)

    model['a_1'] = a_1
    model['a_2'] = a_2
    model['output'] = output

In [ ]:
def get_gradients(model, X_train, Y_train):
    gradients = dict()
    d_output = model['output'] - Y_train
    gradients['d_weight_o'] = np.dot(d_output.T, model['a_2']) / len(Y_train)
    gradients['d_bias_o'] = np.sum(d_output, axis=0) / len(Y_train)

    d_layer2 = np.dot(d_output, model['w_o']) * (model['a_2'] * (1 - model['a_2']))
    gradients['d_weight_2'] = np.dot(d_layer2.T, model['a_1']) / len(Y_train)
    gradients['d_bias_2'] = np.sum(d_layer2, axis=0) / len(Y_train)

    d_layer1 = np.dot(d_layer2, model['w_2']) * (model['a_1'] * (1 - model['a_1']))
    gradients['d_weight_1'] = np.dot(d_layer1.T, X_train) / len(Y_train)
    gradients['d_bias_1'] = np.sum(d_layer1, axis=0) / len(Y_train)
    model['loss'] = cross_entropy_loss(Y_train, model['output'])

    return gradients

In [ ]:
learning_rate = 0.001

def optimize(model, gradients):
    model['w_1'] -= learning_rate * gradients['d_weight_1']
    model['b_1']   -= learning_rate * gradients['d_bias_1']

    model['w_2'] -= learning_rate * gradients['d_weight_2']
    model['b_2']   -= learning_rate * gradients['d_bias_2']

    model['w_o'] -= learning_rate * gradients['d_weight_o']
    model['b_o']   -= learning_rate * gradients['d_bias_o']

In [ ]:
batch_size = 64
epoch = 100
n_loop = len(X_train) / batch_size

model = build_network()

In [ ]:
for i in range(int(n_loop * epoch)):
    # Select a random batch of X_train and Y_train
    indices = np.random.choice(X_train.shape[0], batch_size, replace=False)
    X_batch = X_train[indices]
    Y_batch = Y_train[indices]

    feedforward(model, X_batch)
    gradients = get_gradients(model, X_batch, Y_batch)
    optimize(model, gradients)
    if i % 1000 == 0:  # Print every 100 iterations
        print(f"Iteration {i}, Loss: {model['loss']}")

test out the data


In [ ]:
feedforward(model, X_test)
print("y_test :", y_test[0:12].tolist())
print("preds  :", [int(np.argmax(model['output'][i])) for i in range(12) ])

idx=2
print(model['output'][idx])

In [ ]:
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(X_test_copy[i], cmap='gray', interpolation='none')
    plt.title("Class {}".format(y_test[i]))

plt.tight_layout()

## See if it recognizes my handwriting

#### 글씨를 쓸수 있는 작은 웹앱을 만들자

In [ ]:
from IPython.display import HTML
from google.colab.output import eval_js
from base64 import b64decode

canvas_html = """
<canvas width=%d height=%d></canvas>
<button>Finish</button>
<script>
var canvas = document.querySelector('canvas')
var ctx = canvas.getContext('2d')
ctx.lineWidth = %d
var button = document.querySelector('button')
var mouse = {x: 0, y: 0}

canvas.addEventListener('mousemove', function(e) {
  mouse.x = e.pageX - this.offsetLeft
  mouse.y = e.pageY - this.offsetTop
})
canvas.onmousedown = ()=>{
  ctx.beginPath()
  ctx.moveTo(mouse.x, mouse.y)
  canvas.addEventListener('mousemove', onPaint)
}
canvas.onmouseup = ()=>{
  canvas.removeEventListener('mousemove', onPaint)
}
var onPaint = ()=>{
  ctx.lineTo(mouse.x, mouse.y)
  ctx.stroke()
}

var data = new Promise(resolve=>{
  button.onclick = ()=>{
    resolve(canvas.toDataURL('image/png'))
  }
})
</script>
"""

def draw(filename='drawing.png', w=280, h=280, line_width=20):
  display(HTML(canvas_html % (w, h, line_width)))
  data = eval_js("data")
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return len(binary)

#### 마우스로 숫자를 써서 인식시켜보자

In [ ]:
from PIL import Image
import numpy as np
#from IPython.display import Image
def predict_my_drawing(model):
    draw()
    image = Image.open('drawing.png')

    # summarize some details about the image
    print(image.format, image.size, image.mode)
    image = image.resize((28,28))
    image = image.convert('LA')

    np_frame = np.array(image.getdata())
    digit = np_frame[:,1].reshape(28,28)
    digit_n = digit / 255
    feedforward(model, np.array(digit_n.reshape(-1,784)))
    reco = model['output'][0].reshape(10,-1)
    print(reco)
    print("recognized as", np.argmax(reco))
    return digit

In [ ]:
image = Image.open('drawing.png')

# summarize some details about the image
print(image.format, image.size, image.mode)
image = image.resize((28,28))
image = image.convert('LA')

np_frame = np.array(image.getdata())
digit = np_frame[:,1].reshape(28,28)
digit_n = digit / 255
feedforward(model, np.array(digit_n.reshape(-1,784)))

In [ ]:
digit=predict_my_drawing(model)

In [ ]:
plt.subplot()
plt.imshow(digit, cmap='gray', interpolation='none')


다음 코드에서 target=8로 하여 y_test중에 8자의 이미지를 찾아보자

# sigmoid 와 softmax

## Sigmoid: 어떤 숫자든 0과 1 사이의 값으로 압축하는 함수 (큰 숫자→1에 가깝게, 작은 숫자→0에 가깝게)

```python
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
```

sigmoid 함수는 모든 실수를 0과 1 사이로 변환합니다:
- x가 매우 작은 음수일 때 → 0에 가까워짐
- x = 0일 때 → 정확히 0.5
- x가 매우 큰 양수일 때 → 1에 가까워짐
- x의 절댓값이 클수록 0 또는 1에 빠르게 가까워짐


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.linspace(-10, 10, 200)
y = sigmoid(x)

plt.plot(x, y)
plt.xlabel('x')
plt.ylabel('sigmoid(x)')
plt.title('Sigmoid Function')
plt.grid(True)
plt.show()

## Softmax는 숫자 벡터를 확률 분포로 변환 (모든 값의 합 = 1)

```python
def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return exp_x / np.sum(exp_x)
```

**기본원리**
```python
    exp_x / np.sum(exp_x) # 간단히 표기하면 x / np.sum(x)
```
각 값을 vector 요소의 전체 합으로 나누어 비율(확률)로 만든다

**왜 exp()를 사용하나?**

단순히 x / np.sum(x)를 쓰지 않고 exp(x)를 사용하는 이유:

* 큰 값을 더 강조 (차이 증폭)
* 음수를 양수로 변환 (확률은 0 이상이어야 함)

**왜 최댓값을 빼나?**


```python
    exp_x = np.exp(x - np.max(x))
```

np.exp()는 큰 수에 대해 폭발적으로 커지며 overflow가 발생할 수 있다.

각 숫자를 최댓값 대비 차이로 줄여서 overflow를 방지한다.
(결과는 수학적으로 동일함)


**확률적 의미**
* Squared Error: 단순 거리
    - "예측이 실제와 얼마나 떨어져 있나?"

* Cross Entropy: 정보량
    - "예측 분포가 실제 분포를 얼마나 잘 표현하나?"